Ce notebook a pour objectif de comparer plusieurs modèles de Machine Learning afin d’identifier celui qui prédit le mieux les prix des maisons.

Trois modèles sont testés :

Régression Linéaire

Decision Tree

Random Forest

 1. Chargement des données

Le dataset est rechargé via le pipeline officiel Kaggle afin de garantir la cohérence avec les autres notebooks.

Ensuite, des features supplémentaires issues du feature engineering sont ajoutées :

TotalSF : surface totale

HouseAge : âge de la maison

QualityScore : score global de qualité

 2. Sélection des variables

Les variables utilisées pour la modélisation sont :

surface habitable (GrLivArea)

qualité générale (OverallQual)

nombre de garages (GarageCars)

surface du sous-sol (TotalBsmtSF)

année de construction (YearBuilt)

et les nouvelles features créées

La variable cible est :

SalePrice (prix de la maison)

3. Préparation des données

Les valeurs manquantes sont supprimées pour garantir la stabilité des modèles.

Le dataset est ensuite séparé en :

80% entraînement

20% test

Cela permet d’évaluer les performances des modèles sur des données inconnues.

 4. Entraînement des modèles

Trois modèles sont entraînés et comparés :

 Linear Regression

Modèle simple qui suppose une relation linéaire entre les variables et le prix.

 Decision Tree

Modèle non linéaire capable de capturer des interactions complexes entre variables.

 Random Forest

Ensemble de plusieurs arbres de décision, plus robuste et généralement plus performant.

 5. Évaluation des modèles

Chaque modèle est évalué avec trois métriques :

MAE (Mean Absolute Error) : erreur moyenne absolue

RMSE (Root Mean Squared Error) : erreur quadratique moyenne

R² (coefficient de détermination) : qualité d’explication du modèle

 6. Résultats

Les résultats sont comparés dans un tableau afin d’identifier le modèle le plus performant.

En général :

la régression linéaire est la moins performante

l’arbre de décision améliore les résultats

le Random Forest obtient les meilleures performances

 7. Meilleur modèle

Le meilleur modèle est sélectionné en fonction du score R², qui mesure la capacité du modèle à expliquer la variance des prix.

In [9]:
import pandas as pd
import os
import zipfile
import numpy as np
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ==============================
# LOAD DATASET (OFFICIEL KAGGLE)
# ==============================

os.system(
    "kaggle competitions download -c house-prices-advanced-regression-techniques"
)

with zipfile.ZipFile(
    "house-prices-advanced-regression-techniques.zip",
    "r"
) as zip_ref:
    zip_ref.extractall("data/raw")

df = pd.read_csv("data/raw/train.csv")

# ==============================
# FEATURE ENGINEERING
# ==============================

df["TotalSF"] = df["GrLivArea"] + df["TotalBsmtSF"]
df["HouseAge"] = df["YrSold"] - df["YearBuilt"]
df["QualityScore"] = df["OverallQual"] * df["OverallCond"]

# ==============================
# FEATURES
# ==============================

features = [
    "GrLivArea",
    "OverallQual",
    "GarageCars",
    "TotalBsmtSF",
    "YearBuilt",
    "TotalSF",
    "HouseAge",
    "QualityScore"
]

target = "SalePrice"

# ==============================
# CLEAN DATA
# ==============================

df_model = df[features + [target]].dropna()

X = df_model[features]
y = df_model[target]

# ==============================
# TRAIN TEST SPLIT
# ==============================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ==============================
# MODELS
# ==============================

models = {
    "Linear Regression": LinearRegression(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42
    )
}

# ==============================
# TRAIN & EVALUATE
# ==============================

results = []

for name, model in models.items():

    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    mae = mean_absolute_error(y_test, preds)

    rmse = np.sqrt(mean_squared_error(y_test, preds))

    r2 = r2_score(y_test, preds)

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

# ==============================
# RESULTS
# ==============================

results_df = pd.DataFrame(results)

print(results_df)

# ==============================
# BEST MODEL
# ==============================

best_model = results_df.sort_values(by="R2", ascending=False)

print("\n🏆 Best model:")
print(best_model)

               Model           MAE          RMSE        R2
0  Linear Regression  24440.454590  38962.032705  0.802089
1      Decision Tree  23447.708904  35094.977850  0.839426
2      Random Forest  18055.089064  28470.409020  0.894325

🏆 Best model:
               Model           MAE          RMSE        R2
2      Random Forest  18055.089064  28470.409020  0.894325
1      Decision Tree  23447.708904  35094.977850  0.839426
0  Linear Regression  24440.454590  38962.032705  0.802089
